# InternVL2-8B — evidence first CoT (CoT1_evidence_first)

Replicates Qwen **CoT1** (`../../qwen2p5-3b-7b/all-COT-variations-q7b/evidence-first/`).
Answer-first joint prompt + this CoT structure. Single forward pass per item.

Qwen CoT1 reference: **CI 0.044**  (Qwen best zero-shot: CoT5, CI 0.042).

> **Kaggle:** Settings -> Accelerator -> **GPU T4**. Then Run All.
> Smoke test first: set `MAX_ITEMS = 5` and `SPLIT = 'dev'` in the config cell.

## 1. Install dependencies

In [ ]:
# ── Kaggle setup ──────────────────────────────────────────────────────────────
# transformers is PINNED to 4.49.0, and the pin is load-bearing. Both InternVL2 models
# vendor their own InternLM2ForCausalLM via trust_remote_code, and that class never
# declared GenerationMixin. From v4.50 PreTrainedModel stopped inheriting the mixin, so
# on any newer transformers the language model simply has no .generate() and model.chat()
# dies with:
#     AttributeError: 'InternLM2ForCausalLM' object has no attribute 'generate'
# It fails at the first inference call — i.e. AFTER the model has loaded cleanly.
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

!pip install -q "transformers==4.49.0" accelerate bitsandbytes timm einops \
               sentencepiece "huggingface_hub" 2>&1 | tail -3
import transformers
print(f"Dependencies installed. transformers {transformers.__version__}")

## 2. Configuration

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_ID   = "QCRI/AynVQA-ArabicNLP26"
TASK      = "task1b"
LANG      = "en"
SPLIT     = "devtest"   # "dev" -> labelled, notebook prints CI | "devtest" -> blind, submit to Codabench
MAX_ITEMS = None        # e.g. 5 for a smoke test; None = whole split

RUN_ID    = "CoT1_evidence_first"

VLM_MODEL = "OpenGVLab/InternVL2-8B"
QUANTIZE  = True       # NF4 4-bit — same recipe as the Qwen 7B runs

# InternVL's resolution lever: up to MAX_TILES 448x448 tiles + a thumbnail.
MAX_TILES      = 12
MAX_NEW_TOKENS = 384

# The matching Qwen run, printed next to our score in the scoring cell.
QWEN_REF_NAME = "CoT1"
QWEN_REF      = {'CI': 0.044, 'Comb': 0.956, 'CFHR': 0.0, 'Q+': 0.956, 'Q-': 0.978}

print(f"Run: {RUN_ID}")
print(f"config: {TASK}_{LANG}/{SPLIT} | {VLM_MODEL} | quantized={QUANTIZE}")
print(f"MAX_TILES={MAX_TILES} | MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
print("Comparing against Qwen CoT1 (CI 0.044). Estimated inference: ~45 min on T4")

## 3. Hugging Face login (optional)

In [ ]:
# The AynVQA dataset is public, so anonymous download works and this cell is optional.
# If you hit a rate limit, add a HF READ token as a Kaggle Secret named HF_TOKEN.
# Never hardcode a token in the notebook.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face via Kaggle Secret.")
except Exception as e:
    print(f"No HF_TOKEN secret ({type(e).__name__}) — continuing anonymously (dataset is public).")

## 4. Download the split + images

In [ ]:
import json
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f"{TASK}/{SPLIT}_{LANG}.jsonl", repo_type="dataset")
records = [json.loads(l) for l in open(jsonl, encoding="utf-8") if l.strip()]
if MAX_ITEMS:
    records = records[:MAX_ITEMS]
print(len(records), "items;  labelled:", "labels" in records[0])

# Sequential download — no ThreadPoolExecutor (avoids Kaggle kernel crashes)
needed = sorted({r["image"] for r in records})
paths = {}
for rel in tqdm(needed, desc="images"):
    try:
        paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type="dataset")
    except Exception as e:
        print(f"Failed to download {rel}: {e}")
print(f"Downloaded {len(paths)}/{len(needed)} images.")

## 5. Load InternVL2-8B

In [ ]:
import math
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU — Kaggle: Settings → Accelerator → GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

# T4 (Turing, sm75) has no bf16 and no FlashAttention-2. The InternVL model card
# says bfloat16 + use_flash_attn=True; both of those crash here. fp16 + eager instead.
DTYPE = torch.float16
USE_FLASH_ATTN = False

# ── InternVL image preprocessing: 448px dynamic tiling ────────────────────────
# InternVL has no `max_pixels` knob. It splits the image into up to MAX_TILES
# 448x448 tiles (plus a thumbnail), so MAX_TILES is the resolution lever —
# it is the InternVL analogue of Qwen's MAX_PIXELS.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff, best_ratio = float("inf"), (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff, best_ratio = ratio_diff, ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
         for i in range(1, n + 1) for j in range(1, n + 1)
         if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1])
    ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    target_width, target_height = image_size * ratio[0], image_size * ratio[1]
    blocks = ratio[0] * ratio[1]
    resized = image.resize((target_width, target_height))
    tiles = []
    for i in range(blocks):
        box = ((i % (target_width // image_size)) * image_size,
               (i // (target_width // image_size)) * image_size,
               ((i % (target_width // image_size)) + 1) * image_size,
               ((i // (target_width // image_size)) + 1) * image_size)
        tiles.append(resized.crop(box))
    if use_thumbnail and len(tiles) != 1:
        tiles.append(image.resize((image_size, image_size)))
    return tiles


def load_image(image_file, input_size=448, max_num=MAX_TILES):
    image = Image.open(image_file).convert("RGB")
    transform = build_transform(input_size=input_size)
    tiles = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    return torch.stack([transform(t) for t in tiles])


# ── Load model ────────────────────────────────────────────────────────────────
# Same NF4 double-quant recipe the Qwen 7B runs used, so quantisation is not a
# confound when comparing the two families. device_map={'': 0} pins everything to
# one GPU: 8B at 4-bit is ~5.5 GB and fits a single T4, so no sharding is needed
# and every tensor lands on the same device as the pixel values.
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
) if QUANTIZE else None

load_kwargs = dict(
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    use_flash_attn=USE_FLASH_ATTN,
    trust_remote_code=True,
)
if QUANTIZE:
    model = AutoModel.from_pretrained(
        VLM_MODEL, quantization_config=quant, device_map={"": 0}, **load_kwargs).eval()
else:
    model = AutoModel.from_pretrained(VLM_MODEL, **load_kwargs).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(VLM_MODEL, trust_remote_code=True, use_fast=False)
DEVICE = torch.device("cuda:0")

# Guard for the v4.50 GenerationMixin split. The pin above should already avoid this,
# but if transformers ever resolves to >= 4.50 for a model whose remote code predates
# it, the language model silently loses .generate() and model.chat() dies deep inside
# the vendored code. Re-attach the mixin rather than fail 40 minutes into a run.
from transformers.generation import GenerationMixin

lm_cls = type(model.language_model)
if not issubclass(lm_cls, GenerationMixin):
    lm_cls.__bases__ = (GenerationMixin,) + lm_cls.__bases__
    print(f"Patched {lm_cls.__name__} to inherit GenerationMixin "
          f"(transformers {__import__('transformers').__version__} >= 4.50).")

print(f"Model loaded: {VLM_MODEL} | dtype={DTYPE} | quantized={QUANTIZE} | MAX_TILES={MAX_TILES}")

## 6. Official True/False parser

In [ ]:
# --- True/False parser: verbatim copy of the official backbone.evaluate_tf ---
# This is what the Codabench 1b scorer uses, so local scores match exactly.
import re
from dataclasses import dataclass
from typing import Optional

TRUE_TOKENS = [r"\btrue\b", r"\byes\b", r"\bصح\b", r"\bصحيح\b", r"\bصحيحة\b",
               r"\bالصحيح\b", r"\bالخطأ\b"]
FALSE_TOKENS = [r"\bfalse\b", r"\bfalsy\b", r"\bno\b", r"\bخطأ\b", r"\bالخطأ\b",
                r"\bغلط\b", r"\bغير\s+صحيح(?:ة)?\b", r"\bغير\s+صحيحة\b",
                r"\bخاطئ\b", r"\bخاطئة\b"]
ABSTAIN_PATTERNS = [
    r"\b(can(?:not|'t)\s+determine|can(?:not|'t)\s+tell|not\s+enough\s+information|cannot\s+be\s+sure|unclear)\b",
    r"لا\s+يمكن(?:نا)?\s+الجزم", r"لا\s+يمكن\s+الجزم",
    r"لا\s+يمكن\s+تحديد.*(?:صحة|خطأ|صحيح|خاطئ|العبارة)",
    r"لا\s+نستطيع\s+التأكد", r"لا\s+يمكن\s+الحكم"]
STRONG_CUES = [
    r"therefore[,:\s]*", r"final\s+answer[,:\s]*", r"the\s+answer\s+is[,:\s]*",
    r"correct\s+answer\s+is[,:\s]*", r"so\s+the\s+answer\s+is[,:\s]*",
    r"conclusion[,:\s]*", r"verdict[,:\s]*", r"determination[,:\s]*",
    r"final[,:\s]*(?:answer)?[,:\s]*", r"the\s+statement\s+is\s*[:\-–—,]?\s*",
    r"الإجابة\s+الصحيحة\s*(?:هي)?\s*[:：]?\s*",
    r"الجواب\s+الصحيح\s*(?:هو|هي)?\s*[:：]?\s*",
    r"الإجابة\s*(?:هي)?\s*[:：]?\s*", r"الجواب\s*(?:هو|هي)?\s*[:：]?\s*",
    r"إذًا\s*(?:الجواب|الجواب\s+هو|الإجابة|الإجابة\s+هي)?\s*[:：]?\s*",
    r"الإجابة\s+النهائية\s*(?:هي)?\s*[:：]?\s*"]


@dataclass
class EvalResult:
    pred: Optional[str]
    confidence: float
    needs_review: bool
    reason: str
    conflict: bool


def _normalize(text):
    t = (text or "").strip().replace("‏", "").replace("‎", "").lower()
    return re.sub(r"[ \t]+", " ", t)


def _strip_code_and_quotes(text):
    t = text or ""
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)
    t = re.sub(r"\".*?\"", " ", t, flags=re.DOTALL)
    t = re.sub(r"“.*?”", " ", t, flags=re.DOTALL)
    return t


def _match_label(fragment):
    for pat in TRUE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "true"
    for pat in FALSE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "false"
    return None


def _has_any(patterns, text):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def evaluate_tf(response):
    raw = response or ""
    if not raw.strip():
        return EvalResult(None, 0.0, True, "empty_response", conflict=False)
    text_noquotes = _normalize(_strip_code_and_quotes(raw))

    first_label_pos = first_label_value = None
    for pat, lab in [(p, "true") for p in TRUE_TOKENS] + [(p, "false") for p in FALSE_TOKENS]:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_label_pos is None or m.start() < first_label_pos):
            first_label_pos, first_label_value = m.start(), lab
    first_abstain_pos = None
    for pat in ABSTAIN_PATTERNS:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_abstain_pos is None or m.start() < first_abstain_pos):
            first_abstain_pos = m.start()
    if first_label_pos is not None or first_abstain_pos is not None:
        if first_abstain_pos is not None and (first_label_pos is None or first_abstain_pos < first_label_pos):
            return EvalResult(None, 0.0, True, "abstain_before_label", conflict=False)
        if first_label_pos is not None and (first_abstain_pos is None or first_label_pos < first_abstain_pos):
            return EvalResult(first_label_value, 0.95, False, "first_explicit_label", conflict=False)

    best = None
    m0 = re.match(r"^\s*[\*\s_`]*((?:true|false)|(?:صح|صحيح|صحيحة)|(?:خطأ|غلط|خاطئ|خاطئة))\b",
                  text_noquotes, flags=re.IGNORECASE)
    if m0:
        label = _match_label(m0.group(1))
        if label:
            best = (3.0, label, "leading_label")
    for cue in STRONG_CUES:
        for m in re.finditer(cue, text_noquotes, flags=re.IGNORECASE):
            label = _match_label(text_noquotes[m.end():m.end() + 140])
            if label:
                cand = (3.0, label, "strong_cue")
                best = max(best, cand, key=lambda x: x[0]) if best else cand
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    standalone = [
        (re.compile(r"^[\*\s_`]*true\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "true"),
        (re.compile(r"^[\*\s_`]*false\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "false"),
        (re.compile(r"^[\*\s_`]*صح\s*[\.!\?]*[\*\s_`]*$"), "true"),
        (re.compile(r"^[\*\s_`]*(خطأ|غلط|خاطئ|خاطئة)\s*[\.!\?]*[\*\s_`]*$"), "false")]
    if not best or best[0] < 3.0:
        for ln in reversed(lines[-25:]):
            ln_norm = _normalize(ln)
            for rgx, lab in standalone:
                if rgx.match(ln_norm):
                    best = (2.0, lab, "standalone_label_line")
                    break
            if best:
                break
    if not best:
        tail = text_noquotes[-450:]
        matches = []
        for pat in TRUE_TOKENS:
            matches += [(mm.start(), "true") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        for pat in FALSE_TOKENS:
            matches += [(mm.start(), "false") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        if matches:
            matches.sort(key=lambda x: x[0])
            best = (1.0, matches[-1][1], "last_occurrence_in_tail")
    if not best:
        return EvalResult(None, 0.0, True, "no_label_found", conflict=False)
    score, label, reason = best
    conflict = _has_any(TRUE_TOKENS, text_noquotes) and _has_any(FALSE_TOKENS, text_noquotes)
    confidence = {3.0: 0.95, 2.0: 0.80, 1.0: 0.60}.get(score, 0.50)
    return EvalResult(label, confidence, conflict and score <= 1.0, reason, conflict)

## 7. Prompt + inference

In [ ]:
import re

@torch.no_grad()
def vlm_call(image_path, text, max_new_tokens=MAX_NEW_TOKENS):
    """One InternVL forward pass. Returns the decoded string.

    InternVL takes the image as a pixel_values tensor and requires the '<image>'
    placeholder at the top of the question. Greedy decoding (do_sample=False) to
    match the Qwen runs.
    """
    pixel_values = load_image(image_path).to(DTYPE).to(DEVICE)
    generation_config = dict(max_new_tokens=max_new_tokens, do_sample=False)
    out = model.chat(tokenizer, pixel_values, "<image>\n" + text, generation_config)
    del pixel_values
    torch.cuda.empty_cache()          # release per-item activations -> avoids slow OOM
    return (out or "").strip()

In [ ]:
# Answer-FIRST joint prompt: the model commits to "Answer: X" on line 1, THEN
# justifies. On Qwen this beat reason-first by a wide margin (Run 2 -> Run 4).
ANSWER_SCAN_ORDER = lambda lines: lines[:1] + lines   # first line first, then the rest

PROMPT = (
    "You are a visual fact-checker examining an image from the Arab world.\n"
    "Below are THREE statements. Exactly ONE is grounded in the image (True). "
    "The other two are hallucinations (False).\n\n"
    "Statement 1: {s0}\n"
    "Statement 2: {s1}\n"
    "Statement 3: {s2}\n\n"
    "Instructions:\n"
    "- On the VERY FIRST line write ONLY: \"Answer: X\" where X is 1, 2, or 3.\n"
    "- On the second line write ONE sentence describing only what you literally see "
    "in the image (objects, materials, colours, actions) — do NOT use the statement text.\n"
    "- Then explain why that statement is grounded and the others are not.\n"
    "Do not write anything before the Answer line."
)

In [ ]:
def parse_joint_answer(raw):
    """Parse 'Answer: X' (X in 1..3). Returns 1-indexed int, or None."""
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    for line in ANSWER_SCAN_ORDER(lines):
        m = re.search(r"answer\s*[:\-]?\s*([123])", line, re.IGNORECASE)
        if m:
            return int(m.group(1))
    for line in lines:                      # any standalone digit on its own line
        if re.fullmatch(r"[123]", line):
            return int(line)
    return None


FALLBACK_PROMPT = (
    "You are a visual fact-checker.\n"
    "Decide if the following statement about the image is True or False.\n"
    "Context: exactly one of three statements about this image is True.\n\n"
    "Statement: \"{s}\"\n\n"
    "Write your answer on the FIRST line as exactly one word: True or False.\n"
    "Then briefly explain your reasoning."
)


def predict_item(image_path, statements):
    """Joint prediction for one item. Returns (labels, raw, mode, chosen_1indexed)."""
    raw = vlm_call(image_path, PROMPT.format(s0=statements[0], s1=statements[1], s2=statements[2]))
    chosen = parse_joint_answer(raw)
    if chosen is not None:
        labels = ["false", "false", "false"]
        labels[chosen - 1] = "true"
        return labels, raw, "joint", chosen

    # Fallback: judge each statement on its own. Only reached when the model never
    # emitted an 'Answer: X' line — expected to be rare (<5% on the Qwen runs).
    labels, fb_raws = [], [raw]
    for stmt in statements:
        fb = vlm_call(image_path, FALLBACK_PROMPT.format(s=stmt), max_new_tokens=128)
        fb_raws.append(fb)
        first = fb.splitlines()[0] if fb.strip() else ""
        pred = evaluate_tf(first).pred or evaluate_tf(fb).pred or "false"
        labels.append(pred)
    if labels.count("true") != 1:           # enforce exactly-one-True
        labels = ["true", "false", "false"]
    return labels, " ||| ".join(fb_raws), "fallback", labels.index("true") + 1


print("Inference functions defined.")
print("PROMPT PREVIEW:")
print("-" * 60)
print(PROMPT.format(s0="<statement 1>", s1="<statement 2>", s2="<statement 3>"))
print("-" * 60)

## 8. Run inference

In [ ]:
rows = []       # (id, statement_index, raw, prediction)
results = []    # per-item detail, for the error analysis below
n_joint = n_fallback = 0

for r in tqdm(records, desc=f"[{RUN_ID}] joint infer"):
    labels, raw, mode, chosen = predict_item(paths[r["image"]], r["statements"])
    n_joint    += (mode == "joint")
    n_fallback += (mode == "fallback")
    gold_idx = r["labels"].index(True) if "labels" in r else None
    results.append({
        "id": r["id"], "country": r.get("country", "?"), "category": r.get("category", "?"),
        "gold_idx": gold_idx, "pred_idx": chosen - 1,
        "correct": (chosen - 1 == gold_idx) if gold_idx is not None else None,
        "mode": mode,
    })
    for si, lbl in enumerate(labels):
        rows.append((r["id"], si, raw if si == 0 else "", lbl))

print(f"Done: {len(records)} items | joint: {n_joint} | fallback: {n_fallback}")
print(f"Fallback rate: {n_fallback / len(records) * 100:.1f}%  (Qwen runs: <5%)")

## 9. Write predictions CSV

In [ ]:
import csv
OUT_CSV = f"predictions_{RUN_ID}_{LANG}.csv"
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "statement_index", "raw_prediction", "prediction_parsed"])
    w.writerows(rows)
print("Wrote", OUT_CSV, ":", len(rows), "rows")

## 10. Score (Contrastive Instability)

In [ ]:
gold = {r["id"]: r["labels"].index(True) for r in records if "labels" in r}
if gold:
    by_item = {}
    for iid, si, _raw, parsed in rows:
        by_item.setdefault(iid, {})[si] = parsed

    total = q_plus = q_minus = q_minus_total = combined = 0
    n_partial = n_consistent = 0      # Contrastive Instability
    cfhr_num = cfhr_den = 0           # CFHR
    for iid, true_idx in gold.items():
        total += 1
        q_minus_total += 2
        pr = by_item.get(iid, {})
        labels = {i: evaluate_tf(pr.get(i, "")).pred for i in range(3)}
        ok_t = labels.get(true_idx) == "true"
        ok_f = [labels.get(i) == "false" for i in range(3) if i != true_idx]
        if ok_t:
            q_plus += 1
        q_minus += sum(ok_f)
        all_ok = ok_t and all(ok_f)
        any_ok = ok_t or any(ok_f)
        if all_ok:
            combined += 1
        if any_ok:                    # CI = 1 - consistent / partial
            n_partial += 1
            if all_ok:
                n_consistent += 1
        if ok_t:                      # CFHR = P(miss any Q- | Q+ correct)
            cfhr_den += 1
            if not all(ok_f):
                cfhr_num += 1

    ci           = 1 - n_consistent / n_partial if n_partial else 0.0
    cfhr         = cfhr_num / cfhr_den if cfhr_den else 0.0
    combined_acc = combined / total
    q_plus_acc   = q_plus / total
    q_minus_acc  = q_minus / q_minus_total

    print(f"Run: {RUN_ID}   split: {SPLIT}  ({total} items)")
    print()
    print(f"{'Metric':<34} {'InternVL':>10}   {QWEN_REF_NAME:>14}")
    print("-" * 62)
    print(f"{'Contrastive Instability (CI) v':<34} {ci:>10.4f}   {QWEN_REF['CI']:>14.4f}")
    print(f"{'Combined Accuracy ^':<34} {combined_acc:>10.4f}   {QWEN_REF['Comb']:>14.4f}")
    print(f"{'CFHR v':<34} {cfhr:>10.4f}   {QWEN_REF['CFHR']:>14.4f}")
    print(f"{'Q+ Accuracy ^':<34} {q_plus_acc:>10.4f}   {QWEN_REF['Q+']:>14.4f}")
    print(f"{'Q- Accuracy ^':<34} {q_minus_acc:>10.4f}   {QWEN_REF['Q-']:>14.4f}")
    print()
    delta = ci - QWEN_REF["CI"]
    print(f"CI delta vs {QWEN_REF_NAME}: {delta:+.4f}  "
          f"({'InternVL better' if delta < 0 else 'Qwen better' if delta > 0 else 'tie'})")
else:
    print(f"'{SPLIT}' is blind (no labels) — submit the zip to Codabench for the score.")

## 11. Codabench submission zip

In [ ]:
import zipfile
with open("prediction.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "statement_index", "prediction"])
    for iid, si, _raw, parsed in rows:
        w.writerow([iid, si, parsed])
zip_name = f"prediction_{LANG}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv", "prediction.csv")
print("Wrote", zip_name, " -> submit to Codabench 17051 (task1b English)")
print("Then download it from the Kaggle output and commit it next to this notebook.")